# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Haider-Mughal/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring.**

I'm picking this lane because the starter dataset already shows the shape of the decision I care
about — which pages a reviewer should look at first — and it comes with an honest baseline
(the `stale x visible` hand rule) to beat. The lane guide's own numbers on this exact dataset show
a learned ranking model roughly tripling the baseline's Precision@50 (0.240 -> 0.740), which tells
me there's real signal here worth 7 weeks, not just noise the baseline already captured. It also
plugs directly into the warehouse's daily fact table later, so I can move from the 90-day snapshot
label used here to a proper future-window label (prior 90 days -> next 30 days) without changing
lanes.

In [1]:
# Nothing to check yet — number-backed reasoning for the lane choice is in section 3.


## 2. The question: decision, action, cost of a wrong call

**Question:** Which pages should a content reviewer look at first for a refresh?

**Decision it improves:** which of the thousands of "worth a look" pages a reviewer with limited
hours spends their next hour on.

**Who acts, and what do they do:** an SEO/content reviewer at a FlyRank client. They open the top
of the ranked queue, check the reason codes, and decide to rewrite, expand, re-title, or leave a
page alone.

**Unit of analysis:** one page (one `content_id`) at one snapshot in time. (A future version could
move to page x 30-day-window if I build the warehouse's future-window label.)

**Output:** a ranked queue of pages with a priority score and a reason code per page — not a single
number, so the reviewer can see *why* a page is near the top.

**Cost of a wrong call:**
- *False positive* (page ranked high, nothing was actually wrong): a wasted reviewer-hour — real
  but recoverable cost, since a human still makes the final call before editing anything.
- *False negative* (a genuinely declining, high-traffic page never surfaces near the top): a slower
  cost — lost visibility/traffic keeps compounding until someone happens to notice it another way.
  This is worse than a false positive, which is why Precision@K (not raw accuracy) is the right
  metric — it matches how a capacity-limited reviewer actually consumes the list.

**Why data/ML helps at all:** a single hand-written rule (`stale x visible`) is legible but brittle
— in section 3 it fires on only 17 of 30,000 pages, so on its own it can't fill a reviewer's queue.
The pattern of "worth reviewing" involves several tangled, partially redundant signals (staleness,
visibility, position, CTR, engagement) that interact — exactly the case where a model that can weigh
and combine signals earns its place over a single if-statement, as long as I can still validate it
honestly and read why it ranks a page where it does.

In [2]:
# Nothing to check yet — number-backed reasoning for the cost/decision framing is in section 3.


## 3. Quick look at the data (2-3 real numbers)

*Loading the starter CSV and pulling numbers that motivate this lane: how rare a hand-written rule's
matches are, how strong the signal is when it does fire, and how big the "worth reviewing at all"
pool is that a ranking model actually has to sort through.*

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

n_rows = len(df)
n_clients = df["client_id"].nunique()
base_rate = df["is_declining_label"].mean()

# The starter hand rule from ML-01: stale (no update in 180+ days) AND visible (500+ impressions/90d)
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
stale_visible = stale & visible
n_stale_visible = int(stale_visible.sum())
decline_rate_stale_visible = df.loc[stale_visible, "is_declining_label"].mean()

# How many pages are even worth a reviewer's attention (i.e. actually visible)?
n_worth_reviewing = int((df["impressions_90d"] >= 500).sum())

print(f"Rows: {n_rows:,}  |  Clients: {n_clients}  |  Overall decline rate: {base_rate:.1%}")
print(f"Pages matching the hand rule (stale AND visible): {n_stale_visible} "
      f"({n_stale_visible / n_rows:.2%} of all pages)")
print(f"Decline rate among hand-rule matches: {decline_rate_stale_visible:.1%} "
      f"({decline_rate_stale_visible / base_rate:.2f}x the overall base rate)")
print(f"Pages with >=500 impressions/90d ('visible enough to matter'): {n_worth_reviewing:,} "
      f"({n_worth_reviewing / n_rows:.1%} of all pages)")


Rows: 30,000  |  Clients: 32  |  Overall decline rate: 54.2%
Pages matching the hand rule (stale AND visible): 17 (0.06% of all pages)
Decline rate among hand-rule matches: 94.1% (1.74x the overall base rate)
Pages with >=500 impressions/90d ('visible enough to matter'): 16,726 (55.8% of all pages)


## 4. Careful words: what I can and can't claim

**What this work CAN say, by the end:**
- *Observed:* which signals co-occur with the (proxy) decline label in this snapshot, and how
  strongly.
- *Directional:* whether a learned ranking outperforms the hand rule on Precision@K, under
  client-holdout validation, and by roughly how much.
- *Decision-support:* a ranked queue with reason codes that helps a reviewer spend limited time on
  the most promising candidates first — not a guarantee about any single page.

**What this work CANNOT say:**
- *Not causal:* it cannot claim that refreshing a page **causes** recovery — that needs an
  experiment (e.g. before/after with a control group), which this data doesn't provide.
- *Not about Google's algorithm:* no claims about ranking factors, AI citations, or how search
  engines actually work — only about patterns in FlyRank's own observed metrics.
- *Not a future-outcome label (yet):* the starter target (`trend_direction == "down"`) is a
  same-window proxy, not an observed future outcome. Until I rebuild the label as
  prior-90-days -> next-30-days from the warehouse's daily facts, "declining" here describes the
  current snapshot, not a validated prediction of what happens next.
- *Not free of noise at small N:* the hand rule's 17-page match set (section 3) is too small to
  draw firm conclusions from on its own — it's a starting point to beat, not a finding.

In [4]:
# Careful-words section is text-only by design; the numbers backing it (label proxy, hand-rule N)
# are already computed in section 3.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.